# 01 - Control signals: canny, depth, ControlNet scale

Single product photo (white mug on a dark table), fixed seed 42.
Goal: understand what `controlnet_conditioning_scale` actually does for canny, depth and both together.

**Findings**
- Low scale (0.2) loses the silhouette entirely; the model drifts to the prompt's generic object.
- Canny at scale >= 0.8 invents a gold rim: the double edge line at the mug's mouth forces the model to draw *something* there. Depth never produces it.
- Depth at scale >= 0.6 flattens the marble texture; the depth map treats surface detail as noise.
- Multi-ControlNet at 1.0/1.0 is not broken but sterile, and the gold rim is suppressed: controls can cancel each other's artifacts.
- Working point: **canny 0.8 + depth 0.3**.

Two problems carried forward to notebook 03: ControlNet preserves geometry but not identity, and the product keeps the lighting of the original photo.

In [ ]:
import os, sys
sys.path.insert(0, "../src")
# run from notebooks/; switch to the repo root so relative paths work
os.chdir("..")
print("working dir:", os.getcwd())

## 1. Canny edges

Classic thresholds first, then a sweep. The mug's shadowed left edge stays open at every threshold pair.

In [ ]:
import cv2
import numpy as np
from PIL import Image

image = Image.open("inputs/testset/01_matte_mug.jpeg").resize((512, 640))
array = np.array(image)

edges = cv2.Canny(array, 100, 200)
Image.fromarray(edges).save("outputs/canny_100_200.png")
print("saved canny_100_200.png")

In [ ]:
for low, high in [(50, 100), (100, 200), (200, 300)]:
    edges = cv2.Canny(array, low, high)
    Image.fromarray(edges).save(f"outputs/canny_{low}_{high}.png")
    print(f"canny_{low}_{high}.png")

**Fix: blur first, then lower the thresholds.** Blur suppresses random noise while consistent but weak edges survive, so the threshold can safely come down. Cost: the shadow boundary on the table is picked up as an edge too - canny cannot tell an object boundary from a lighting boundary.

In [ ]:
blurred = cv2.GaussianBlur(array, (5, 5), 0)

for low, high in [(20, 60), (30, 80), (50, 120)]:
    edges = cv2.Canny(blurred, low, high)
    Image.fromarray(edges).save(f"outputs/canny_blur_{low}_{high}.png")
    print(f"canny_blur_{low}_{high}.png")

Final recipe: blur (5,5) + thresholds (20, 60), stacked to 3 channels because ControlNet expects RGB.

In [ ]:
edges = cv2.Canny(blurred, 20, 60)
edges_rgb = np.stack([edges] * 3, axis=-1)
Image.fromarray(edges_rgb).save("outputs/canny_final.png")
print("canny_final.png ready")

## 2. Depth (MiDaS)

Canny encodes lines, depth encodes mass. They are complementary: the shadowed left edge is clear in depth, the mug/table contact line is clear in canny, and the busy cabinet in the background collapses into a single far plane in depth.

In [ ]:
from controlnet_aux import MidasDetector

midas = MidasDetector.from_pretrained("lllyasviel/Annotators")
depth = midas(image).resize((512, 640))
depth.save("outputs/depth_final.png")
print("depth_final.png ready")

## 3. ControlNet pipeline and a first generation

fp16 is required on 12 GB. UniPC gives good results in 20 steps, which matters when producing grids.

In [ ]:
import torch
from diffusers import (StableDiffusionControlNetPipeline, ControlNetModel,
                       UniPCMultistepScheduler)

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", controlnet=controlnet,
    torch_dtype=torch.float16, safety_checker=None)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda")
print("pipeline ready")

In [ ]:
PROMPT = ("professional product photo of a white ceramic mug on a marble "
          "kitchen counter, soft studio lighting, high quality")
NEGATIVE = "blurry, low quality, distorted, deformed"

canny_image = Image.open("outputs/canny_final.png")
generator = torch.Generator(device="cuda").manual_seed(42)

result = pipe(prompt=PROMPT, negative_prompt=NEGATIVE, image=canny_image,
              num_inference_steps=20, controlnet_conditioning_scale=1.0,
              generator=generator, height=640, width=512).images[0]
result.save("outputs/first_generation_scale_1.0.png")

## 4. Canny scale grid (0.2 - 1.4)

The generator is reset **inside** the loop so all seven runs start from the same noise; any difference between images is then attributable to the scale alone.

In [ ]:
SCALES = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4]

for scale in SCALES:
    generator = torch.Generator(device="cuda").manual_seed(42)
    result = pipe(prompt=PROMPT, negative_prompt=NEGATIVE, image=canny_image,
                  num_inference_steps=20, controlnet_conditioning_scale=scale,
                  generator=generator, height=640, width=512).images[0]
    result.save(f"outputs/grid_canny_scale_{scale}.png")
    print(f"scale={scale} done")

## 5. Depth grid

Free VRAM first: `del` drops the reference, `gc.collect()` collects it, `empty_cache()` returns PyTorch's reserved pool.

In [ ]:
import gc

del pipe, controlnet
gc.collect()
torch.cuda.empty_cache()
print(f"allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
controlnet_depth = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-depth", torch_dtype=torch.float16)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", controlnet=controlnet_depth,
    torch_dtype=torch.float16, safety_checker=None)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda")

depth_image = Image.open("outputs/depth_final.png")
for scale in SCALES:
    generator = torch.Generator(device="cuda").manual_seed(42)
    result = pipe(prompt=PROMPT, negative_prompt=NEGATIVE, image=depth_image,
                  num_inference_steps=20, controlnet_conditioning_scale=scale,
                  generator=generator, height=640, width=512).images[0]
    result.save(f"outputs/grid_depth_scale_{scale}.png")
    print(f"depth scale={scale} done")

## 6. Multi-ControlNet

`controlnet`, `image` and `conditioning_scale` all become lists, and the order must match across them - a mismatch fails silently. Instead of a full 7x7 sweep, seven informative combinations.

In [ ]:
del pipe, controlnet_depth
gc.collect()
torch.cuda.empty_cache()

controlnet_canny = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16)
controlnet_depth = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-depth", torch_dtype=torch.float16)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=[controlnet_canny, controlnet_depth],
    torch_dtype=torch.float16, safety_checker=None)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda")
print("multi-ControlNet ready")

In [ ]:
COMBOS = [
    (1.0, 1.0),  # both at full strength - tests the "over-constrained" hypothesis
    (0.7, 0.7),
    (0.5, 0.5),
    (0.8, 0.3),  # canny dominant: sharp silhouette, light volume support
    (0.3, 0.8),  # depth dominant
    (0.6, 0.2),
    (0.2, 0.6),
]

for canny_scale, depth_scale in COMBOS:
    generator = torch.Generator(device="cuda").manual_seed(42)
    result = pipe(prompt=PROMPT, negative_prompt=NEGATIVE,
                  image=[canny_image, depth_image],
                  controlnet_conditioning_scale=[canny_scale, depth_scale],
                  num_inference_steps=20, generator=generator,
                  height=640, width=512).images[0]
    result.save(f"outputs/grid_multi_c{canny_scale}_d{depth_scale}.png")
    print(f"canny={canny_scale} depth={depth_scale} done")

## 7. Contact sheets

One image per grid, for the README.

In [ ]:
import glob

def contact_sheet(pattern, out_path, columns=4, tile=(256, 320)):
    files = sorted(glob.glob(pattern))
    images = [Image.open(f).resize(tile) for f in files]
    rows = (len(images) + columns - 1) // columns
    sheet = Image.new("RGB", (columns * tile[0], rows * tile[1]), "white")
    for i, img in enumerate(images):
        sheet.paste(img, ((i % columns) * tile[0], (i // columns) * tile[1]))
    sheet.save(out_path)
    print(f"{out_path}: {len(images)} images")

contact_sheet("outputs/grid_canny_scale_*.png", "outputs/sheet_canny.png")
contact_sheet("outputs/grid_depth_scale_*.png", "outputs/sheet_depth.png")
contact_sheet("outputs/grid_multi_*.png", "outputs/sheet_multi.png")